<a href="https://colab.research.google.com/github/Jatindeswal/Vynix/blob/main/Vynix_Colab_Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎓 PROJECT VYNIX: Few-Shot Human-Object Interaction (HOI) Detector
### *Spatial Scene Graph Fusion with Geometric Hallucination Veto*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jatindeswal/Vynix/blob/main/Vynix_Colab_Demo.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-Repository-black?logo=github)](https://github.com/Jatindeswal/Vynix)

---

## 📌 Executive Abstract & Problem Statement
Modern Vision-Language Models (such as OpenAI CLIP) suffer from **spatial and physical hallucinations** in compositional reasoning tasks: they frequently predict active physical interactions (e.g., *"person holding a cup"*) merely because a human and an object co-occur in the same scene, even when they are physically separate.

**Project Vynix** introduces a lightweight, four-stage zero-shot/few-shot HOI architecture that couples **YOLOv8 object localization**, **pure spatial graph geometry**, and **CLIP contrastive alignment** with a novel **Vynix Logic Gate**:
$$\mathcal{V}(v, \text{IoU}) = \begin{cases} \text{"No Interaction (Proximity Only)"} & \text{if } v \in \mathcal{C}_{\text{contact}} \;\land\; \text{IoU} = 0.0 \\ \operatorname{argmax}(\text{CLIP}) & \text{otherwise} \end{cases}$$

This notebook provides a complete interactive walkthrough designed for demonstration and academic evaluation.


## ⚙️ Step 0: Environment Setup & Dependencies
Install the required packages (`ultralytics` for YOLOv8, `transformers` for CLIP, `opencv`, `pillow`, `matplotlib`).


In [1]:
# Install required computer vision and deep learning packages
!pip install ultralytics transformers opencv-python Pillow torch torchvision matplotlib --quiet

import os, sys, math, io, urllib.request
import cv2
import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from ultralytics import YOLO
from transformers import CLIPModel, CLIPProcessor

print("✓ All libraries imported successfully!")
print(f"✓ PyTorch version: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✓ Active Compute Device: {device}")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 84.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.2/64.2 kB 7.6 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✓ All libraries imported successfully!
✓ PyTorch version: 2.11.0+cu128 | CUDA Available: True
✓ Active Compute Device: cuda


## 📥 Step 1: Download Benchmark Test Images
We download two evaluation test images from the official Vynix GitHub repository:
- `test_image.jpg`: **Ground Truth = Person HOLDING a cup** (Physical contact, $\text{IoU} > 0$)
- `test_image1.jpg`: **Ground Truth = Person STANDING NEAR a cup** (No contact, $\text{IoU} = 0$)


In [ ]:
# Download test benchmark images directly from GitHub
repo_raw = "https://raw.githubusercontent.com/Jatindeswal/Vynix/main"
for filename in ["test_image.jpg", "test_image1.jpg"]:
    if not os.path.exists(filename):
        print(f"Downloading {filename}...")
        urllib.request.urlretrieve(f"{repo_raw}/{filename}", filename)
        print(f"✓ Saved {filename}")
    else:
        print(f"✓ {filename} already present.")


## 👁️ Stage 1: Node Extraction ("The Eyes")
We utilize **YOLOv8-nano** to extract localized bounding boxes $[x_1, y_1, x_2, y_2]$ for:
1. **Human Agent** ($c = 0$, `person`)
2. **Object Patient** ($c = 41$, `cup` or any COCO target class)


In [ ]:
# Initialize YOLOv8-nano model
yolo_model = YOLO("yolov8n.pt")

def extract_nodes(image_path, person_cls=0, object_cls=41, conf=0.25):
    """
    Detects person and object bounding boxes in the input image.

    Args:
        image_path (str): Path to input RGB image.
        person_cls (int): COCO class ID for human agent (0 = person).
        object_cls (int): COCO class ID for target object (41 = cup).
        conf (float): Confidence threshold for detections.

    Returns:
        tuple: (person_box, object_box) in [x1, y1, x2, y2] pixel coordinates.
    """
    results = yolo_model(image_path, device=device, verbose=False)
    pbox, obox = None, None
    for r in results:
        for i in range(len(r.boxes)):
            cls_id = int(r.boxes.cls[i].item())
            score = float(r.boxes.conf[i].item())
            if score < conf:
                continue
            box = r.boxes.xyxy[i].tolist()
            if cls_id == person_cls and pbox is None:
                pbox = box
            elif cls_id == object_cls and obox is None:
                obox = box
            if pbox is not None and obox is not None:
                return pbox, obox
    return pbox, obox


## 📐 Stage 2: Scene Graph Math ("The Brain")
Pure geometric calculations to model spatial relations without model bias:
1. **Centroids**: Centre coordinate of each bounding box $C = \left(\frac{x_1+x_2}{2}, \frac{y_1+y_2}{2}\right)$
2. **Euclidean Distance**: $L_2$ distance between centroids $d = \sqrt{(x_h - x_o)^2 + (y_h - y_o)^2}$
3. **Intersection-over-Union (IoU)**: Spatial bounding box overlap $\text{IoU} = \frac{\text{Area}(B_h \cap B_o)}{\text{Area}(B_h \cup B_o)}$
4. **Union Bounding Box**: $B_{\text{union}} = [\min(x_1), \min(y_1), \max(x_2), \max(y_2)]$ used as the VLM semantic context window.


In [ ]:
def compute_centroid(box):
    """Calculates the geometric centre of an axis-aligned bounding box."""
    return ((box[0] + box[2]) / 2.0, (box[1] + box[3]) / 2.0)

def compute_euclidean_distance(pt1, pt2):
    """Calculates the L2 Euclidean distance between two 2D points in pixel space."""
    return math.sqrt((pt1[0] - pt2[0])**2 + (pt1[1] - pt2[1])**2)

def compute_iou(box_a, box_b):
    """
    Calculates the Intersection-over-Union (IoU) overlap between two bounding boxes.
    Returns 0.0 for physically disjoint bounding boxes.
    """
    ix1, iy1 = max(box_a[0], box_b[0]), max(box_a[1], box_b[1])
    ix2, iy2 = min(box_a[2], box_b[2]), min(box_a[3], box_b[3])
    inter_area = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)

    area_a = (box_a[2] - box_a[0]) * (box_a[3] - box_a[1])
    area_b = (box_b[2] - box_b[0]) * (box_b[3] - box_b[1])
    union_area = area_a + area_b - inter_area

    return inter_area / union_area if union_area > 0 else 0.0

def compute_union_box(box_a, box_b):
    """Calculates the minimum enclosing bounding box enclosing both entities."""
    return [min(box_a[0], box_b[0]), min(box_a[1], box_b[1]),
            max(box_a[2], box_b[2]), max(box_a[3], box_b[3])]


## 🗣️ Stage 3: Vision-Language Alignment ("The Translator")
We use **OpenAI CLIP (ViT-B/32)** to classify the interaction semantics within the union crop region against competing interaction hypotheses:
- *Hypothesis 1*: `"a person holding a cup"` (Active Contact)
- *Hypothesis 2*: `"a person standing near a cup but not touching it"` (Non-Contact Proximity)


In [ ]:
# Initialize CLIP Processor and Model
clip_model_name = "openai/clip-vit-base-patch32"
print(f"Loading CLIP ({clip_model_name})...")
clip_processor = CLIPProcessor.from_pretrained(clip_model_name)
clip_model = CLIPModel.from_pretrained(clip_model_name).to(device).eval()

CANDIDATE_PROMPTS = [
    "a person holding a cup",
    "a person standing near a cup but not touching it"
]

def vlm_classify(image_path, ubox, prompts=CANDIDATE_PROMPTS):
    """
    Crops the union bounding box, passes it to CLIP, and computes softmax probabilities.
    """
    img = cv2.imread(image_path)
    h, w = img.shape[:2]
    x1, y1 = max(0, int(ubox[0])), max(0, int(ubox[1]))
    x2, y2 = min(w, int(ubox[2])), min(h, int(ubox[3]))

    # Crop and convert BGR (OpenCV) to RGB (PIL)
    crop_rgb = cv2.cvtColor(img[y1:y2, x1:x2], cv2.COLOR_BGR2RGB)
    pil_crop = Image.fromarray(crop_rgb)

    inputs = clip_processor(text=prompts, images=pil_crop, return_tensors="pt", padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = clip_model(**inputs).logits_per_image # Shape: (1, num_prompts)
        probs = torch.softmax(logits, dim=1).squeeze(0).cpu().tolist()

    return dict(zip(prompts, probs))


## 🛡️ Stage 4: The Vynix Logic Gate ("The Arbiter")
### Mathematical Formalization
Let $v^* = \operatorname{argmax}_{v} P_{\text{CLIP}}(v \mid I_{\text{union}})$ be the top-predicted interaction prompt from CLIP.

If $v^*$ belongs to the **Contact Interaction Set** $\mathcal{C}_{\text{contact}}$ (e.g. *holding*, *carrying*, *drinking*), physical contact requires non-zero bounding box overlap.

$$\text{If } v^* \in \mathcal{C}_{\text{contact}} \quad \text{and} \quad \text{IoU}(B_h, B_o) = 0.0 \implies \text{Physical contact is impossible}$$

The **Vynix Gate** overrides the VLM's hallucinated prediction to the null/non-contact state.


In [ ]:
CONTACT_KEYWORDS = ["holding", "carrying", "drinking", "eating", "riding", "touching"]

def vynix_logic_gate(vlm_probs, iou_val):
    """
    Fuses vision-language semantic predictions with spatial-geometric evidence.

    Returns:
        tuple: (final_verdict, gate_status, explanation)
    """
    top_prompt = max(vlm_probs, key=vlm_probs.get)
    is_contact_interaction = any(kw in top_prompt.lower() for kw in CONTACT_KEYWORDS)

    # Trigger condition: VLM predicts contact, but geometric IoU is 0.0
    if is_contact_interaction and iou_val == 0.0:
        overridden_verdict = "a person standing near a cup but not touching it"
        gate_status = "OVERRIDE [VETO TRIGGERED]"
        explanation = (
            "VLM hallucination detected: CLIP predicted contact ('holding'), but bounding "
            "box overlap IoU is exactly 0.00% (disjoint). Physically grounded veto applied."
        )
        return overridden_verdict, gate_status, explanation
    else:
        gate_status = "PASS [TRUST VLM]"
        explanation = (
            "Semantic prediction is geometrically consistent with spatial coordinates."
        )
        return top_prompt, gate_status, explanation


## 🎨 Visual Proof Annotation Engine
Renders color-coded bounding boxes:
- **Person (Human Agent)**: Blue bounding box $(255, 120, 50)$
- **Cup (Target Object)**: Green bounding box $(50, 220, 50)$
- **Metric Overlay Banner**: Displays calculated $\text{IoU}$ percentage and final grounded verdict.


In [ ]:
def render_proof_image(image_path, pbox, obox, iou_val, verdict, gate_status):
    """Annotates the input image with bounding boxes, IoU, and final verdict."""
    img = cv2.imread(image_path)

    # 1. Draw Person Bounding Box (Blue)
    cv2.rectangle(img, (int(pbox[0]), int(pbox[1])), (int(pbox[2]), int(pbox[3])), (255, 120, 50), 3)
    cv2.putText(img, "Person", (int(pbox[0]), int(pbox[1]) - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 120, 50), 2)

    # 2. Draw Cup Bounding Box (Green)
    cv2.rectangle(img, (int(obox[0]), int(obox[1])), (int(obox[2]), int(obox[3])), (50, 220, 50), 3)
    cv2.putText(img, "Cup", (int(obox[0]), int(obox[1]) - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (50, 220, 50), 2)

    # 3. Draw Dark Overlay Banner at Top
    cv2.rectangle(img, (0, 0), (img.shape[1], 70), (25, 25, 25), -1)

    banner_color = (0, 255, 255) if "OVERRIDE" in gate_status else (0, 255, 100)
    cv2.putText(img, f"IoU: {iou_val*100:.1f}%  |  Gate: {gate_status}",
                (15, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.65, banner_color, 2)
    cv2.putText(img, f"Verdict: {verdict}",
                (15, 55), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    # Convert BGR to RGB for matplotlib display
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


## 🧪 Step 5: Execute Two-Image Benchmark & Run Pipeline
We now process both benchmark images end-to-end and extract full mathematical metrics.


In [ ]:
benchmark_suite = [
    {
        "name": "Case A: Physical Contact Interaction",
        "file": "test_image.jpg",
        "ground_truth": "Person HOLDING a cup"
    },
    {
        "name": "Case B: Visual Proximity without Contact",
        "file": "test_image1.jpg",
        "ground_truth": "Person STANDING NEAR a cup (Not holding)"
    }
]

execution_records = []

for case in benchmark_suite:
    img_path = case["file"]

    # Stage 1: Detection
    pbox, obox = extract_nodes(img_path)
    if pbox is None or obox is None:
        print(f"❌ Detection failed for {img_path}")
        continue

    # Stage 2: Spatial Math
    p_center = compute_centroid(pbox)
    o_center = compute_centroid(obox)
    dist_px = compute_euclidean_distance(p_center, o_center)
    iou_val = compute_iou(pbox, obox)
    ubox = compute_union_box(pbox, obox)

    # Stage 3: VLM Semantic Classification
    vlm_probs = vlm_classify(img_path, ubox)

    # Stage 4: Vynix Logic Gate
    verdict, gate_status, explanation = vynix_logic_gate(vlm_probs, iou_val)

    # Render Visual Proof
    annotated_img = render_proof_image(img_path, pbox, obox, iou_val, verdict, gate_status)

    execution_records.append({
        "case": case["name"],
        "ground_truth": case["ground_truth"],
        "file": img_path,
        "pbox": pbox,
        "obox": obox,
        "p_center": p_center,
        "o_center": o_center,
        "dist_px": dist_px,
        "iou_val": iou_val,
        "ubox": ubox,
        "vlm_probs": vlm_probs,
        "raw_top": max(vlm_probs, key=vlm_probs.get),
        "verdict": verdict,
        "gate_status": gate_status,
        "explanation": explanation,
        "annotated_img": annotated_img
    })

print("✓ Benchmark execution complete for all cases!")


## 🖼️ Side-by-Side Visual Comparison
Displaying the visual proofs generated by the Vynix pipeline.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 9))

for idx, rec in enumerate(execution_records):
    axes[idx].imshow(rec["annotated_img"])
    axes[idx].set_title(f"{rec['case']}\nGT: {rec['ground_truth']}", fontsize=13, fontweight='bold')
    axes[idx].axis("off")

plt.tight_layout()
plt.show()


## 📊 Quantitative Output & Viva Evaluation Board
### Detailed Mathematical Breakdown for Viva Defense & Faculty Review


In [ ]:
for idx, r in enumerate(execution_records, 1):
    print("=" * 80)
    print(f"  📌 BENCHMARK EVALUATION REPORT — CASE {idx}: {r['case'].upper()}")
    print("=" * 80)

    print(f"  • Source Image        : {r['file']}")
    print(f"  • Ground Truth Label  : {r['ground_truth']}")
    print("-" * 80)

    print("  [1] DETECTED BOUNDING BOXES (Stage 1: YOLOv8-nano):")
    print(f"      - Person Box [x1,y1,x2,y2] : {[round(v, 1) for v in r['pbox']]}")
    print(f"      - Cup Box    [x1,y1,x2,y2] : {[round(v, 1) for v in r['obox']]}")
    print(f"      - Union Crop [x1,y1,x2,y2] : {[round(v, 1) for v in r['ubox']]}")

    print("\n  [2] SPATIAL GRAPH METRICS (Stage 2: Geometric Brain):")
    print(f"      - Person Centroid (Cx, Cy) : ({r['p_center'][0]:.1f} px, {r['p_center'][1]:.1f} px)")
    print(f"      - Cup Centroid    (Cx, Cy) : ({r['o_center'][0]:.1f} px, {r['o_center'][1]:.1f} px)")
    print(f"      - Centroid Distance (L2)   : {r['dist_px']:.2f} pixels")
    print(f"      - Spatial Overlap (IoU)    : {r['iou_val'] * 100:.2f}%  (IoU scalar: {r['iou_val']:.4f})")

    print("\n  [3] VLM ALIGNMENT SCORES (Stage 3: OpenAI CLIP ViT-B/32):")
    for prompt, prob in r['vlm_probs'].items():
        bar = "█" * int(prob * 30)
        print(f"      - "{prompt}" : {prob*100:6.2f}% | {bar}")
    print(f"      → Raw CLIP Argmax Winner   : "{r['raw_top']}"")

    print("\n  [4] VYNIX LOGIC GATE ARBITRATION (Stage 4: Grounded Logic Gate):")
    print(f"      - Gate Decision Status     : {r['gate_status']}")
    print(f"      - Scientific Justification : {r['explanation']}")
    print("-" * 80)
    print(f"  🏆 FINAL GROUNDED VERDICT      : "{r['verdict']}"")
    print("=" * 80)
    print("\n")
